# AUTOMATIZACIÓN DE CORRESPONDENCIA - AFILIADOS SIN SISBÉN IV

Este notebook gestiona la correspondencia masiva mensual para notificar a los afiliados activos del Régimen Subsidiado en **CAPRESOCA EPS** que no cuentan con la encuesta del Sisbén IV o clasificación por listado censal.

## Lógica de Negocio y Arquitectura Corporativa
- **Requisito Normativo**: Decreto 780 de 2016 (Artículo 2.1.5.1.3 y concordantes).
- **Identidad Institucional**: Se reutilizan los recursos visuales y logos corporativos oficiales del OneDrive de la EPS.
- **Desacoplamiento**: El flujo está separado en dos fases independientes: Generación de PDFs Consolidados (Fase 1) y Envío de Correos mediante Outlook (Fase 2).
- **Validación del Dato**: Filtros vectorizados estrictos para identificar radicados duplicados, correos electrónicos con estructura inválida, inconsistencias territoriales con la base DANE, y registros sin identificación.
- **Modo Prueba**: Control de seguridad global mediante `MODO_PRUEBA = True` para evitar envíos masivos por error.

# 1. IMPORTACIÓN DE MÓDULOS

Se importan las librerías necesarias para el análisis de datos, automatización de oficina, generación de PDFs y auditoría.

In [ ]:
import pandas as pd  # Manipulación y análisis de datos
import numpy as np  # Operaciones numéricas y nulos
import win32com.client as win32  # Automatización de Outlook en Windows
import os  # Operaciones del sistema de archivos
import gc  # Recolección de basura para optimización de RAM
import time  # Control de tiempos y throttling de envíos
import re  # Expresiones regulares para validaciones de correos
import unicodedata  # Normalización de textos y eliminación de acentos
import pdfkit  # Generador de PDFs a partir de HTML (requiere wkhtmltopdf)
from pathlib import Path  # Rutas de archivos multiplataforma
from datetime import datetime  # Manejo de fechas y marcas de tiempo
import traceback  # Trazabilidad completa de excepciones

print("Módulos importados correctamente.")

# 2. CONFIGURACIÓN GENERAL Y VARIABLES DE CONTROL

Definición de cuentas de origen, variables temporales, parámetros mensuales y activación del **Modo Prueba** para evitar reprocesos y envíos involuntarios.

In [ ]:
# ==========================================
# CONFIGURACIÓN DE IDENTIDAD Y SEGURIDAD
# ==========================================

# MODO PRUEBA: Si está activo, limita el procesamiento a una muestra de 3 registros
# y NO envía correos reales a los afiliados (simula y guarda borrador si aplica)
MODO_PRUEBA: bool = False

CUENTA_ORIGEN: str = "procesosbdua@capresoca.gov.co"
NOMBRE_MOSTRAR: str = "ASEGURAMIENTO REGIMEN SUBSIDIADO"
CORREO_VENTANILLA_INTERNA: str = "capresocaeps@capresoca-casanare.gov.co"

# Control mensual del proceso
ANO_PROCESO: str = "2026"
CONTRATO: str = "CTO 102.2026"
INFORME_NUM: str = "06"
MES_TEXTO: str = "Junio"

# Fechas personalizadas para visualización en PDF (Vacío "" usa fecha actual)
FECHA_ENVIO_DOCUMENTO: str = "22 de junio de 2026"  # Ej: "2 de marzo de 2026"
FECHA_ENVIO_OUTLOOK: str = "lunes, 22 de junio de 2026 04:17 PM"    # Ej: "martes, 2 de marzo de 2026 05:00 PM"

# Ruta externa del motor PDF (Wkhtmltopdf)
PATH_WKHTML_EXE: Path = Path(r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe")

# Identificación de OneDrive y Directorios del Proyecto
USER_HOME: Path = Path.home()
ONEDRIVE_PATH: Path = USER_HOME / "OneDrive - 891856000_CAPRESOCA E P S"

# Construcción de ruta base con OneDrive
RUTA_PROYECTO_BASE: Path = (
    ONEDRIVE_PATH / 
    "Escritorio" / 
    "Yesid Rincón Z" / 
    "informes" / 
    ANO_PROCESO / 
    CONTRATO / 
    f"{CONTRATO.replace(' ', '')} Informe  #{INFORME_NUM}" / 
    "12 Actividad" / 
    "Bases de datos notificaciones telefonicas" / 
    "USUARIOS ACTIVOS SIN SISBEN (IV)"
)

# Rutas de entrada de datos e imágenes corporativas
FILE_EXCEL_PATH: Path = RUTA_PROYECTO_BASE / "No Sisben-06-05-2026.xlsx"
RUTA_DEPARTAMENTOS_TXT: Path = ONEDRIVE_PATH / "Capresoca" / "AlmostClear" / "Constantes" / "Departamentos.txt"
RUTA_IMG_CAPRE: Path = ONEDRIVE_PATH / "Imágenes" / "capre.png"
RUTA_IMG_SIPER: Path = ONEDRIVE_PATH / "Imágenes" / "Siper salud.png"

# Rutas de salida estructuradas
FOLDER_SALIDA: Path = RUTA_PROYECTO_BASE / "Resultados"
FOLDER_PDF_MUNICIPIOS: Path = FOLDER_SALIDA / "PDF_MUNICIPIOS"
FOLDER_TEMP_HTML: Path = FOLDER_SALIDA / "TEMP_HTML"
FOLDER_TEMP_PDF: Path = FOLDER_SALIDA / "TEMP_PDF"
FOLDER_LOGS: Path = FOLDER_SALIDA / "LOGS"

# Creación de directorios para asegurar la integridad de la ejecución
for folder in [FOLDER_SALIDA, FOLDER_PDF_MUNICIPIOS, FOLDER_TEMP_HTML, FOLDER_TEMP_PDF, FOLDER_LOGS]:
    folder.mkdir(parents=True, exist_ok=True)

# Validaciones previas de integridad del entorno
assert PATH_WKHTML_EXE.exists(), f"ERROR: wkhtmltopdf no encontrado en {PATH_WKHTML_EXE}"
assert FILE_EXCEL_PATH.exists(), f"ERROR: Excel de entrada no encontrado en {FILE_EXCEL_PATH}"
assert RUTA_DEPARTAMENTOS_TXT.exists(), f"ERROR: Catálogo DANE no encontrado en {RUTA_DEPARTAMENTOS_TXT}"
assert RUTA_IMG_CAPRE.exists(), f"ERROR: Logo de Capresoca no encontrado en {RUTA_IMG_CAPRE}"
assert RUTA_IMG_SIPER.exists(), f"ERROR: Logo de Supersalud no encontrado en {RUTA_IMG_SIPER}"

print(f"Entorno configurado correctamente.")
print(f"Modo Prueba: {'ACTIVADO' if MODO_PRUEBA else 'DESACTIVADO'}")
print(f"Directorio de resultados listo en: {FOLDER_SALIDA.name}")

del USER_HOME
gc.collect()

# 3. CARGA DE ARCHIVOS Y MAPEO DE MUNICIPIOS (DANE)

Carga del archivo de afiliados e importación de la base municipal de la EPS (`Departamentos.txt`) para cruzar y homologar códigos territoriales.

In [ ]:
# 1. Carga de Excel principal
try:
    df_raw = pd.read_excel(
        FILE_EXCEL_PATH, 
        sheet_name='Datos', 
        dtype=str,
        engine='openpyxl'
    )
    # Limpieza básica de headers (eliminar espacios invisibles)
    df_raw.columns = [str(col).strip() for col in df_raw.columns]
    print(f"Excel cargado: {len(df_raw)} registros leídos de la hoja 'Datos'")
except Exception as e:
    print(f"Error crítico al leer el archivo Excel: {e}")
    raise

# 2. Función de normalización de municipios para resolver inconsistencias
def clean_name(name):
    if not isinstance(name, str):
        return ""
    name = name.upper().strip()
    # Descomponer unicode para eliminar acentos de manera segura
    nfkd_form = unicodedata.normalize('NFKD', name)
    cleaned = "".join(c for c in nfkd_form if 'A' <= c <= 'Z' or c.isspace())
    return " ".join(cleaned.split())

# 3. Carga y parsing del catálogo de Departamentos (DANE)
dane_muni_map = {}
try:
    with open(RUTA_DEPARTAMENTOS_TXT, 'r', encoding='utf-8') as f:
        header = f.readline().strip().split(';')
        for line in f:
            parts = line.strip().split(';')
            if len(parts) >= 4 and parts[1].strip().upper() == 'CASANARE':
                raw_name = parts[3].strip()
                code = parts[2].strip()  # Código corto de 3 dígitos
                cleaned_name = clean_name(raw_name)
                dane_muni_map[cleaned_name] = code
    print(f"Catálogo DANE cargado: {len(dane_muni_map)} municipios homologados para Casanare")
except Exception as e:
    print(f"Error crítico al leer Departamentos.txt: {e}")
    raise

gc.collect()

# 4. VALIDACIONES DE INTEGRIDAD Y FILTRADO (CALIDAD DEL DATO)

Aplicación de filtros de calidad vectorizados. Se aíslan registros sin identificación, radicados vacíos o duplicados, correos electrónicos erróneos e inconsistencias geográficas en un reporte detallado.

In [ ]:
# Estructuras para capturar el resultado de las validaciones
rejected_records = []
valid_records = []

# Patrón simple para correo electrónico
email_regex = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

# Conjuntos para control de duplicados
radicados_vistos = set()
afiliados_vistos = set()

# Inicializar contadores de métricas
count_radicado_vacio = 0
count_identificacion_invalida = 0
count_municipio_no_homologado = 0
count_correo_invalido = 0
count_radicado_duplicado = 0
count_afiliado_duplicado = 0

print("Iniciando ciclo de validaciones fila por fila...")

for idx, row in df_raw.iterrows():
    fila_excel = idx + 2  # Convertir a índice basado en 1 y considerando cabecera
    
    radicado = str(row.get('Radicado', '')).strip()
    tipo_doc = str(row.get('4', '')).strip()
    num_doc = str(row.get('5', '')).strip()
    nombre_muni_raw = str(row.get('Nombre Municipio', '')).strip()
    correo = str(row.get('correo_electronico', '')).strip()
    
    # 1. Filtro Obligatorio: Radicado no nulo/vacío
    if not radicado or radicado.lower() in ['nan', 'null', '']:
        count_radicado_vacio += 1
        # Se descarta en silencio según requerimiento principal de Aseguramiento
        continue
        
    razones_rechazo = []
    
    # 2. Validación: Datos de Identificación
    if not tipo_doc or not num_doc or tipo_doc.lower() == 'nan' or num_doc.lower() == 'nan':
        razones_rechazo.append("IDENTIFICACION_INCOMPLETA")
        count_identificacion_invalida += 1

    # 3. Validación: Nomenclatura Territorial y Homologación DANE
    muni_limpio = clean_name(nombre_muni_raw)
    mnc_id = dane_muni_map.get(muni_limpio, None)
    if not mnc_id:
        razones_rechazo.append(f"MUNICIPIO_NO_HOMOLOGADO ('{nombre_muni_raw}')")
        count_municipio_no_homologado += 1
        
    # 4. Validación: Estructura del correo electrónico
    if not correo or correo.lower() == 'nan' or not email_regex.match(correo):
        razones_rechazo.append(f"CORREO_INVALIDO ('{correo}')")
        count_correo_invalido += 1
        
    # 5. Validación: Radicado duplicado
    if radicado in radicados_vistos:
        razones_rechazo.append(f"RADICADO_DUPLICADO ('{radicado}')")
        count_radicado_duplicado += 1
    else:
        if not razones_rechazo: # Solo registrar si va por buen camino
            radicados_vistos.add(radicado)
            
    # 6. Validación: Afiliado duplicado en el mismo lote
    llave_afiliado = (tipo_doc, num_doc)
    if tipo_doc and num_doc:
        if llave_afiliado in afiliados_vistos:
            razones_rechazo.append(f"AFILIADO_REPETIDO ({tipo_doc} {num_doc})")
            count_afiliado_duplicado += 1
        else:
            if not razones_rechazo:
                afiliados_vistos.add(llave_afiliado)

    # Clasificación final del registro
    if razones_rechazo:
        rechazo_info = {
            'Fila_Excel': fila_excel,
            'Identificacion': f"{tipo_doc} {num_doc}",
            'Municipio': nombre_muni_raw,
            'Radicado': radicado,
            'Etapa_Falla': "VALIDACION_DATOS",
            'Excepcion': ", ".join(razones_rechazo),
            'Detalle': str(row.to_dict())
        }
        rejected_records.append(rechazo_info)
    else:
        # Datos extendidos con código DANE y municipio normalizado
        datos_validos = row.to_dict()
        datos_validos['MNC_ID'] = mnc_id
        datos_validos['Municipio_Limpio'] = muni_limpio
        datos_validos['MUNICIPIO_GRUPO'] = f"{mnc_id}_{muni_limpio}"
        valid_records.append(datos_validos)

df_procesar = pd.DataFrame(valid_records)
df_rechazados = pd.DataFrame(rejected_records)

print(f"\n--- AUDITORÍA DE CALIDAD ---")
print(f"Total registros leídos: {len(df_raw)}")
print(f"Registros omitidos (sin Radicado): {count_radicado_vacio}")
print(f"Registros rechazados por fallas en validación: {len(df_rechazados)}")
print(f"Registros válidos aptos para generación: {len(df_procesar)}")

del radicados_vistos, afiliados_vistos
gc.collect()

# 5. NORMALIZACIÓN DE COLUMNAS E IDENTIDAD BDUA

Traducción de los nombres de columnas de formato numérico original a las convenciones institucionales BDUA para alimentar la plantilla HTML y sanitizar variables vacías.

In [ ]:
if not df_procesar.empty:
    # Mapeo de columnas numéricas del Excel original a nombres estándar
    column_mapping = {
        '4': 'TPS_IDN_ID',
        '5': 'HST_IDN_NUMERO_IDENTIFICACION',
        '6': 'AFL_PRIMER_APELLIDO',
        '7': 'AFL_SEGUNDO_APELLIDO',
        '8': 'AFL_PRIMER_NOMBRE',
        '9': 'AFL_SEGUNDO_NOMBRE',
        '10': 'Fecha Nacimiento',
        '11': 'Sexo',
        '42': 'Regimen',
        'Nombre Municipio': 'Nombre_Municipio'
    }
    
    df_procesar.rename(columns=column_mapping, inplace=True)
    
    # Sanitización de variables del destinatario para evitar campos nulos o NaNs
    for col in ['AFL_PRIMER_APELLIDO', 'AFL_SEGUNDO_APELLIDO', 'AFL_PRIMER_NOMBRE', 'AFL_SEGUNDO_NOMBRE']:
        if col in df_procesar.columns:
            df_procesar[col] = df_procesar[col].fillna('').astype(str).str.strip().str.upper()
            
    # Asegurar mayúsculas en correos y nombres
    df_procesar['correo_electronico'] = df_procesar['correo_electronico'].str.strip().str.lower()
    df_procesar['Nombre_Municipio'] = df_procesar['Nombre_Municipio'].str.strip().str.upper()
    
    # Construcción de nombre completo del afiliado
    df_procesar['Nombre_Completo'] = (
        df_procesar['AFL_PRIMER_NOMBRE'] + " " +
        df_procesar['AFL_SEGUNDO_NOMBRE'] + " " +
        df_procesar['AFL_PRIMER_APELLIDO'] + " " +
        df_procesar['AFL_SEGUNDO_APELLIDO']
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    print("Columnas del DataFrame procesar normalizadas a estándar BDUA:")
    print(df_procesar.columns.tolist())
else:
    print("ADVERTENCIA: No hay registros válidos para normalizar.")

gc.collect()

# 6. DEFINICIÓN DE PLANTILLA HTML INSTITUCIONAL

Se incrusta la plantilla HTML con la identidad corporativa y se adapta el mensaje del documento `No sisben.docx` eliminando plazos límites estrictos para lograr una notificación general informativa.

In [ ]:
# Definición de la plantilla de correo HTML corporativa
# Los logos se referencian con 'cid:' para la incrustación directa en Outlook
# y se reemplazarán con rutas absolutas locales al generar el PDF independiente.

HTML_TEMPLATE: str = """
<html>
<body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333; margin: 20px;">
    <table width="100%" style="border-collapse: collapse;">
        <tr>
            <td style="width: 30%; text-align: left;">
                <img src="cid:logo_capre" alt="Capresoca EPS" width="180">
            </td>
            <td style="width: 70%; text-align: right; font-size: 10px; color: #666;">
                NIT. 891.856.000-7<br>
                EPS en intervención<br>
                COMUNICACIÓN INSTITUCIONAL<br>
                FO-GD-01 | 2024-01-19 | V.03
            </td>
        </tr>
    </table>

    <hr style="border: 0; border-top: 1px solid #0056b3;">

    <p style="text-align: left;"><strong>AEI/120:</strong> {Radicado}</p>
    
    <p>Yopal-Casanare, {Fecha_Envio}</p>

    <div style="margin-top: 20px; font-size: 14px;">
        <strong>Señor(a):</strong><br>
        {Nombre_Completo}<br>
        {TPS_IDN_ID}: {HST_IDN_NUMERO_IDENTIFICACION}<br>
        {MUNICIPIO_GRUPO}<br>
        {correo_electronico}
    </div>

    <p style="margin-top: 20px;"><strong>Ref.:</strong> {Asunto}</p>

    <p>Cordial saludo,</p>

    <p>Desde <strong>Capresoca EPS</strong> trabajamos diariamente para garantizar la continuidad y calidad en la prestación de sus servicios de salud. Revisando nuestros registros, hemos identificado que usted se encuentra como afiliado activo en el Régimen Subsidiado, pero a la fecha no registra información en la ficha de caracterización socioeconómica del Sisbén en su última metodología (Sisbén IV), ni se encuentra reportado(a) como población especial a través de un listado censal.</p>

    <p>De acuerdo con lo establecido en la normatividad vigente del sector salud (Decreto 780 de 2016), es indispensable contar con la clasificación del Sisbén para determinar la permanencia en el Régimen Subsidiado. Por lo tanto, le recordamos la importancia de adelantar este trámite para evitar novedades futuras que puedan afectar el estado de su afiliación.</p>

    <p><strong>¿Qué debe hacer?</strong> Le recomendamos acercarse a la mayor brevedad a la oficina del Sisbén de la Alcaldía de su municipio en <strong>{Nombre_Municipio}</strong> y solicitar la aplicación de la encuesta bajo la última metodología vigente. Mantener su información socioeconómica actualizada es un deber clave para garantizar que usted y su grupo familiar sigan protegidos sin interrupciones en sus tratamientos y atenciones médicas.</p>

    <p>Para cualquier consulta adicional o recibir acompañamiento en este proceso, no dude en ponerse en contacto con nosotros a través de nuestros canales de atención al cliente. Estamos disponibles para asistirle en el número telefónico celular: <strong>3106805416</strong> o en el correo electrónico institucional: <a href="mailto:aseguramiento@capresoca-casanare.gov.co" style="color: #0056b3;">aseguramiento@capresoca-casanare.gov.co</a>.</p>
    
    <div style="margin-top: 40px;">
        Atentamente,<br><br><br>
        <strong>Osmar Yesid Rincón Zorro</strong><br>
        Profesional de apoyo área de aseguramiento
    </div>

    <table width="100%" style="margin-top: 50px; border-top: 2px solid #0056b3; font-size: 11px; color: #444;">
        <tr>
            <td style="width: 40%; text-align: left; padding-top: 15px;">
                <img src="cid:logo_siper" alt="SuperSalud" width="200">
            </td>
            <td style="width: 60%; text-align: right; padding-top: 15px; line-height: 1.2;">
                Calle 7 No. 19 – 34. Línea de atención gratuita nacional: 018000912880<br>
                Email. capresocaeps@capresoca-casanare.gov.co<br>
                Yopal - Casanare &nbsp;&nbsp;&nbsp; Página 1 de 1
            </td>
        </tr>
    </table>
</body>
</html>
"""

print("Plantilla HTML registrada en memoria para mapeo dinámico.")
gc.collect()

# 7. FASE 1: GENERACIÓN DOCUMENTAL Y CONSOLIDACIÓN DE PDFs

Renderización y almacenamiento de la documentación en `/TEMP_HTML` e `/TEMP_PDF` de forma independiente al motor de correos de Outlook. Generación de PDFs consolidados geográficamente por municipio.

In [ ]:
print("Iniciando Fase 1: Generación Documental...")

# Lista para acumular los logs operativos de la Fase 1
pdf_generation_logs = []
pdf_errors = []

# Filtrado para Modo Prueba en caso de estar activo
if MODO_PRUEBA:
    df_lote = df_procesar.head(3).copy()
    print(f"[MODO PRUEBA ACTIVO] Limitando procesamiento a una muestra de {len(df_lote)} registros.")
else:
    df_lote = df_procesar.copy()

if df_lote.empty:
    print("ADVERTENCIA: No hay registros para procesar en esta fase.")
else:
    # Configuración de pdfkit
    config_pdf = pdfkit.configuration(wkhtmltopdf=str(PATH_WKHTML_EXE))
    
    PDF_OPTIONS = {
        'page-size': 'Letter',
        'margin-top': '0.5in',
        'margin-right': '0.5in',
        'margin-bottom': '0.5in',
        'margin-left': '0.5in',
        'encoding': "UTF-8",
        'enable-local-file-access': None,
        'quiet': ''
    }
    
    # Agrupar por MUNICIPIO_GRUPO
    municipios_grupos = df_lote.groupby('MUNICIPIO_GRUPO')
    
    for nombre_grupo, grupo in municipios_grupos:
        print(f"\nProcesando grupo territorial: {nombre_grupo} ({len(grupo)} afiliados)")
        html_acumulado_muni = ""
        
        # Ruta del PDF consolidado por municipio final
        ruta_pdf_salida_consolidado = FOLDER_PDF_MUNICIPIOS / f"{nombre_grupo}.pdf"
        
        for idx, row in grupo.iterrows():
            datos_afiliado = row.to_dict()
            # Campos adicionales de fecha de envío (Usa variable configurable o fecha del sistema)
            datos_afiliado['Fecha_Envio'] = FECHA_ENVIO_DOCUMENTO if FECHA_ENVIO_DOCUMENTO else datetime.now().strftime("%d de %B de %Y")
            
            # Cabecera simulada de correo para incorporar al PDF impreso (Usa variable configurable o fecha del sistema)
            fecha_actual_pdf = FECHA_ENVIO_OUTLOOK if FECHA_ENVIO_OUTLOOK else datetime.now().strftime("%A, %d de %B de %Y %I:%M %p")
            header_outlook_pdf = f"""
            <div style="font-family: 'Segoe UI', Tahoma, sans-serif; color: #000; margin-bottom: 30px;">
                <h2 style="margin: 0; padding: 0; font-size: 20px; font-weight: bold;">{NOMBRE_MOSTRAR}</h2>
                <hr style="border: 0; border-top: 2.5px solid #000; margin: 5px 0 15px 0;">
                <table width="100%" style="border-collapse: collapse; font-size: 13px; line-height: 1.6;">
                    <tr><td style="width: 85px; font-weight: bold; color: #666;">De:</td><td style="font-weight: bold;">{NOMBRE_MOSTRAR} &lt;{CUENTA_ORIGEN}&gt;</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Enviado el:</td><td>{fecha_actual_pdf}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Para:</td><td>{row['correo_electronico']}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Asunto:</td><td style="font-weight: bold;">{row['Asunto']}</td></tr>
                </table>
            </div>
            """
            
            # 1. Reemplazo de datos de afiliado en la plantilla HTML
            try:
                html_afiliado = HTML_TEMPLATE.format(**datos_afiliado)
                
                # Guardar HTML temporal de depuración
                ruta_html_temp = FOLDER_TEMP_HTML / f"{row['HST_IDN_NUMERO_IDENTIFICACION']}.html"
                with open(ruta_html_temp, 'w', encoding='utf-8') as f_html:
                    f_html.write(html_afiliado)
                
                # Reemplazo de cids por imágenes reales locales para el renderizador PDF
                html_para_pdf = (header_outlook_pdf + html_afiliado).replace("cid:logo_capre", str(RUTA_IMG_CAPRE))
                html_para_pdf = html_para_pdf.replace("cid:logo_siper", str(RUTA_IMG_SIPER))
                
                # Acumulación de páginas para consolidación municipal con salto de página
                html_acumulado_muni += f'<div style="page-break-after: always; padding: 10px;">{html_para_pdf}</div>'
                
                # Registrar estado operativo exitoso por afiliado en esta etapa
                pdf_generation_logs.append({
                    'HST_IDN_NUMERO_IDENTIFICACION': row['HST_IDN_NUMERO_IDENTIFICACION'],
                    'MUNICIPIO_GRUPO': nombre_grupo,
                    'Radicado': row['Radicado'],
                    'HTML_Generado': True,
                    'PDF_Individual_Temporal': True,
                    'Estado': 'HTML_PREPARADO'
                })
            except Exception as e_html:
                tb_resumido = traceback.format_exc().splitlines()[-2:]
                pdf_errors.append({
                    'Fila_Excel': idx + 2,
                    'Identificacion': row.get('HST_IDN_NUMERO_IDENTIFICACION', 'SN'),
                    'Municipio': nombre_grupo,
                    'Radicado': row.get('Radicado', 'SN'),
                    'Etapa_Falla': 'GENERACION_HTML',
                    'Excepcion': str(e_html),
                    'Detalle': " | ".join(tb_resumido)
                })
                print(f"   [ERROR HTML] ID {row.get('HST_IDN_NUMERO_IDENTIFICACION', 'SN')}: {e_html}")
        
        # 2. Renderizado del PDF Consolidado del Municipio
        if html_acumulado_muni:
            try:
                pdfkit.from_string(
                    html_acumulado_muni, 
                    str(ruta_pdf_salida_consolidado), 
                    configuration=config_pdf, 
                    options=PDF_OPTIONS
                )
                print(f"   [PDF OK] Consolidado municipal generado: {ruta_pdf_salida_consolidado.name}")
                
                # Actualizar estado de los logs operativos para confirmar PDF consolidado generado
                for log in pdf_generation_logs:
                    if log['MUNICIPIO_GRUPO'] == nombre_grupo:
                        log['PDF_Consolidado_Generado'] = True
                        log['Estado'] = 'PDF_GENERADO'
                        
            except Exception as e_pdf:
                tb_resumido = traceback.format_exc().splitlines()[-2:]
                pdf_errors.append({
                    'Fila_Excel': 'Grupo Consolidado',
                    'Identificacion': nombre_grupo,
                    'Municipio': nombre_grupo,
                    'Radicado': 'CONSOLIDADO',
                    'Etapa_Falla': 'GENERACION_PDF_CONSOLIDADO',
                    'Excepcion': str(e_pdf),
                    'Detalle': " | ".join(tb_resumido)
                })
                print(f"   [ERROR PDF] Falló generación consolidada para {nombre_grupo}: {e_pdf}")
                # Marcar como fallidos en el log operativo
                for log in pdf_generation_logs:
                    if log['MUNICIPIO_GRUPO'] == nombre_grupo:
                        log['PDF_Consolidado_Generado'] = False
                        log['Estado'] = 'ERROR_PDF'

df_logs_fase1 = pd.DataFrame(pdf_generation_logs)
df_errors_fase1 = pd.DataFrame(pdf_errors)
print(f"\nFase 1 completada. PDFs Consolidados generados: {len(df_logs_fase1['MUNICIPIO_GRUPO'].unique()) if not df_logs_fase1.empty else 0}")
gc.collect()

# 8. FASE 2: PREPARACIÓN Y AUTOMATIZACIÓN DE CORREOS (OUTLOOK)

Conexión con el servicio corporativo de Outlook a través de la librería `win32com`. Preparación y enrutamiento del mensaje incrustando los logos institucionales mediante protocolo CID.

In [ ]:
print("Iniciando Fase 2: Automatización de Correos...")

email_logs = []
email_errors = []

if df_lote.empty:
    print("ADVERTENCIA: No hay registros para procesar en Fase 2.")
else:
    try:
        # Inicialización de la sesión local de Outlook
        outlook = win32.Dispatch("Outlook.Application")
        namespace = outlook.GetNamespace("MAPI")
        
        # Detección y enrutamiento forzado a través de la cuenta origen autorizada de Capresoca
        account = next((a for a in namespace.Accounts if a.SmtpAddress.lower() == CUENTA_ORIGEN.lower()), None)
        if not account:
            raise ValueError(f"Error Crítico: La cuenta de correo origen '{CUENTA_ORIGEN}' no está configurada en este Outlook.")
            
        print(f"Sesión Outlook vinculada exitosamente a la cuenta: {CUENTA_ORIGEN}")
        
        # Procesamiento fila a fila del lote
        for idx, row in df_lote.iterrows():
            identificacion = row['HST_IDN_NUMERO_IDENTIFICACION']
            destinatario_principal = row['correo_electronico']
            asunto = row['Asunto']
            
            # Lógica de copia (BCC) a Ventanilla Única obligatoria
            bcc_final = CORREO_VENTANILLA_INTERNA
            
            # Cargar HTML renderizado temporalmente para este afiliado en el paso anterior
            ruta_html_temp = FOLDER_TEMP_HTML / f"{identificacion}.html"
            
            if not RUTA_IMG_CAPRE.exists() or not RUTA_IMG_SIPER.exists():
                raise FileNotFoundError("No se encontraron las imágenes institucionales para incrustar en el correo.")
            
            if not ruta_html_temp.exists():
                # Si el HTML temporal no existe por fallo previo en Fase 1, se registra el error y se salta
                email_errors.append({
                    'Fila_Excel': idx + 2,
                    'Identificacion': identificacion,
                    'Municipio': row['MUNICIPIO_GRUPO'],
                    'Radicado': row['Radicado'],
                    'Etapa_Falla': 'CARGA_HTML_CORREO',
                    'Excepcion': 'HTML_NO_ENCONTRADO',
                    'Detalle': f"Falta el HTML temporal en la ruta: {ruta_html_temp.name}"
                })
                print(f"   [SKIPPED] No existe HTML para ID {identificacion}. Generación documental previa fallida.")
                continue
                
            with open(ruta_html_temp, 'r', encoding='utf-8') as f_in:
                html_body_mail = f_in.read()
                
            try:
                # Crear objeto de correo electrónico de Outlook
                mail = outlook.CreateItem(0)
                # Comando de enrutamiento interno de bajo nivel de Outlook para cuenta específica
                mail._oleobj_.Invoke(*(64209, 0, 8, 0, account))
                
                mail.Subject = asunto
                mail.To = destinatario_principal
                mail.BCC = bcc_final
                
                # Adjuntar imágenes y configurar cabeceras CID corporativas para visualización offline
                for img_id, path_img in [("logo_capre", RUTA_IMG_CAPRE), ("logo_siper", RUTA_IMG_SIPER)]:
                    att = mail.Attachments.Add(str(path_img))
                    att.PropertyAccessor.SetProperty("http://schemas.microsoft.com/mapi/proptag/0x3712001E", img_id)
                
                mail.HTMLBody = html_body_mail
                
                # Acción dependiente del Modo de Seguridad Prueba
                if MODO_PRUEBA:
                    # En modo prueba NO llamamos a mail.Send(), guardamos como borrador para auditoría visual
                    mail.Save() 
                    print(f"   [TEST BORRADOR] Correo preparado en Borradores para ID {identificacion} -> Destino: {destinatario_principal}")
                    estado_proceso = 'CORREO_PREPARADO_PRUEBA'
                else:
                    # Envío real a producción
                    mail.Send()
                    print(f"   [ENVIO REAL OK] Correo transmitido con éxito para ID {identificacion} -> {destinatario_principal}")
                    estado_proceso = 'CORREO_ENVIADO'
                    
                email_logs.append({
                    'HST_IDN_NUMERO_IDENTIFICACION': identificacion,
                    'Correo_Preparado': True,
                    'Correo_Destino': destinatario_principal,
                    'Estado_Envio': estado_proceso
                })
                
                # Throttling preventivo: Retraso de 1.2 segundos para evitar bloqueos y control de velocidad en Exchange Server
                time.sleep(1.2)
                
            except Exception as e_send:
                tb_resumido = traceback.format_exc().splitlines()[-2:]
                email_errors.append({
                    'Fila_Excel': idx + 2,
                    'Identificacion': identificacion,
                    'Municipio': row['MUNICIPIO_GRUPO'],
                    'Radicado': row['Radicado'],
                    'Etapa_Falla': 'ENVIO_CORREO_OUTLOOK',
                    'Excepcion': str(e_send),
                    'Detalle': " | ".join(tb_resumido)
                })
                print(f"   [ERROR ENVIO] Falló el despacho de correo para ID {identificacion}: {e_send}")
                
    except Exception as e_outlook_init:
        print(f"ERROR CRÍTICO: No se pudo inicializar la API de Outlook: {e_outlook_init}")
        raise

df_logs_fase2 = pd.DataFrame(email_logs)
df_errors_fase2 = pd.DataFrame(email_errors)
gc.collect()

# 9. GENERACIÓN DE REPORTES DE TRAZABILIDAD (LOGS AVANZADOS)

Generación y exportación de reportes detallados del proceso corporativo en formato Excel: `LOG_OPERACION.xlsx`, `LOG_ERRORES.xlsx` y `LOG_RESUMEN.xlsx`.

In [ ]:
print("Iniciando consolidación de logs y exportación de reportes...")

# 1. CREACIÓN DE LOG_OPERACION
# Combinamos los resultados de la generación documental (Fase 1) y de los correos (Fase 2)
if not df_procesar.empty:
    # Lote base de afiliados que pasaron filtros iniciales
    df_operacion = df_procesar[[
        'HST_IDN_NUMERO_IDENTIFICACION', 'Nombre_Completo', 'MUNICIPIO_GRUPO', 'Radicado', 'correo_electronico'
    ]].copy()
    
    # Cruce con Logs de Generación de PDF
    if 'df_logs_fase1' in locals() and not df_logs_fase1.empty:
        df_operacion = df_operacion.merge(
            df_logs_fase1[['HST_IDN_NUMERO_IDENTIFICACION', 'HTML_Generado', 'PDF_Consolidado_Generado']],
            on='HST_IDN_NUMERO_IDENTIFICACION', 
            how='left'
        )
    else:
        df_operacion['HTML_Generado'] = False
        df_operacion['PDF_Consolidado_Generado'] = False
        
    # Cruce con Logs de Envíos de Correo
    if 'df_logs_fase2' in locals() and not df_logs_fase2.empty:
        df_operacion = df_operacion.merge(
            df_logs_fase2[['HST_IDN_NUMERO_IDENTIFICACION', 'Correo_Preparado', 'Estado_Envio']],
            on='HST_IDN_NUMERO_IDENTIFICACION', 
            how='left'
        )
    else:
        df_operacion['Correo_Preparado'] = False
        df_operacion['Estado_Envio'] = 'NO_PROCESADO_FASE_2'
        
    # Rellenar valores vacíos generados por cruces incompletos o exclusión de lotes de prueba
    df_operacion['HTML_Generado'] = df_operacion['HTML_Generado'].fillna(False)
    df_operacion['PDF_Consolidado_Generado'] = df_operacion['PDF_Consolidado_Generado'].fillna(False)
    df_operacion['Correo_Preparado'] = df_operacion['Correo_Preparado'].fillna(False)
    df_operacion['Estado_Envio'] = df_operacion['Estado_Envio'].fillna('NO_PROCESADO_PRUEBA')
    
    # Renombrar para presentación corporativa
    df_operacion.rename(columns={
        'HST_IDN_NUMERO_IDENTIFICACION': 'Identificacion_Afiliado',
        'Nombre_Completo': 'Nombre_Afiliado',
        'MUNICIPIO_GRUPO': 'Municipio_Geografico',
        'Radicado': 'Radicado_Notificacion',
        'correo_electronico': 'Correo_Destino'
    }, inplace=True)
else:
    df_operacion = pd.DataFrame(columns=[
        'Identificacion_Afiliado', 'Nombre_Afiliado', 'Municipio_Geografico', 
        'Radicado_Notificacion', 'Correo_Destino', 'HTML_Generado', 
        'PDF_Consolidado_Generado', 'Correo_Preparado', 'Estado_Envio'
    ])

# Exportación de LOG_OPERACION
ruta_log_operacion = FOLDER_LOGS / "LOG_OPERACION.xlsx"
df_operacion.to_excel(ruta_log_operacion, index=False, engine='openpyxl')
print(f"LOG_OPERACION exportado: {ruta_log_operacion.name} ({len(df_operacion)} registros registrados)")


# 2. CREACIÓN DE LOG_ERRORES
# Unimos los errores de validación previa, errores de fase 1 y errores de fase 2
lista_errores_consolidada = []

# Incorporar errores de validación de datos del bloque 5
if not df_rechazados.empty:
    lista_errores_consolidada.append(df_rechazados)
    
# Incorporar errores de renderizado de Fase 1
if 'df_errors_fase1' in locals() and not df_errors_fase1.empty:
    lista_errores_consolidada.append(df_errors_fase1)
    
# Incorporar errores de envío de Fase 2
if 'df_errors_fase2' in locals() and not df_errors_fase2.empty:
    lista_errores_consolidada.append(df_errors_fase2)

if lista_errores_consolidada:
    df_errores_final = pd.concat(lista_errores_consolidada, ignore_index=True)
else:
    df_errores_final = pd.DataFrame(columns=[
        'Fila_Excel', 'Identificacion', 'Municipio', 'Radicado', 'Etapa_Falla', 'Excepcion', 'Detalle'
    ])
    
# Exportación de LOG_ERRORES
ruta_log_errores = FOLDER_LOGS / "LOG_ERRORES.xlsx"
df_errores_final.to_excel(ruta_log_errores, index=False, engine='openpyxl')
print(f"LOG_ERRORES exportado: {ruta_log_errores.name} ({len(df_errores_final)} fallas documentadas)")


# 3. CREACIÓN DE LOG_RESUMEN
conceptos_resumen = [
    ('Marca de Tiempo del Reporte', datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    ('Modo de Seguridad Prueba Activo', str(MODO_PRUEBA)),
    ('Total Filas Excel Leidas', len(df_raw)),
    ('Registros Omitidos (Sin Radicado)', count_radicado_vacio),
    ('Registros Validos para Generar', len(df_procesar)),
    ('Registros Excluidos por Errores de Validacion', len(df_rechazados)),
    ('Fallas en Identificacion (Tipo/Num Vacios)', count_identificacion_invalida),
    ('Fallas Territoriales (Municipio No Homologado)', count_municipio_no_homologado),
    ('Fallas en Direccionamiento (Correo Invalido)', count_correo_invalido),
    ('Excluidos por Duplicación de Radicado', count_radicado_duplicado),
    ('Excluidos por Afiliado Repetido', count_afiliado_duplicado),
    ('PDFs Consolidados por Municipio Generados', len(df_operacion[df_operacion['PDF_Consolidado_Generado'] == True]['Municipio_Geografico'].unique())),
    ('Correos Preparados / Borradores en Outlook', len(df_operacion[df_operacion['Correo_Preparado'] == True])),
    ('Errores de Generación Documental (PDF)', len(df_errors_fase1) if 'df_errors_fase1' in locals() else 0),
    ('Errores de Envío por Outlook', len(df_errors_fase2) if 'df_errors_fase2' in locals() else 0),
    ('Municipios Casanare Procesados con Exito', ", ".join(df_procesar['Municipio_Limpio'].unique()) if not df_procesar.empty else 'Ninguno')
]

df_resumen = pd.DataFrame(conceptos_resumen, columns=['Metrica_Control', 'Valor'])

# Exportación de LOG_RESUMEN
ruta_log_resumen = FOLDER_LOGS / "LOG_RESUMEN.xlsx"
df_resumen.to_excel(ruta_log_resumen, index=False, engine='openpyxl')
print(f"LOG_RESUMEN exportado: {ruta_log_resumen.name} listo para reporte directivo")

gc.collect()

# 10. MÉTRICAS FINALES DE LA AUTOMATIZACIÓN

Visualización en pantalla del informe final de ejecución corporativa para auditoría directa del operador.

In [ ]:
print(f"=========================================================")
print(f"       INFORME DE CONTROL OPERACIONAL - CAPRESOCA EPS    ")
print(f"=========================================================")
print(f" Lote procesado: {MES_TEXTO} {ANO_PROCESO} | Contrato: {CONTRATO}")
print(f" Modo Prueba: {'ACTIVO (Ejecución de simulacion)' if MODO_PRUEBA else 'DESACTIVADO (Producción real)'}")
print(f"---------------------------------------------------------")
print(f" Total registros leídos del Excel original:  {len(df_raw)}")
print(f" Registros válidos que pasaron filtros:      {len(df_procesar)}")
print(f" Registros rechazados por calidad del dato:  {len(df_rechazados)}")
print(f" Registros excluidos (sin Radicado):         {count_radicado_vacio}")
print(f"---------------------------------------------------------")
print(f" desglose de fallas encontradas:")
print(f"   * Identificaciones incompletas:          {count_identificacion_invalida}")
print(f"   * Municipios sin código DANE:            {count_municipio_no_homologado}")
print(f"   * Correos con sintaxis inválida:         {count_correo_invalido}")
print(f"   * Radicados duplicados en origen:        {count_radicado_duplicado}")
print(f"   * Afiliados duplicados en el lote:       {count_afiliado_duplicado}")
print(f"---------------------------------------------------------")
if 'df_logs_fase1' in locals() and not df_logs_fase1.empty:
    pdf_ok_count = len(df_logs_fase1[df_logs_fase1['PDF_Consolidado_Generado'] == True])
    print(f" PDFs consolidados geográficamente generados: {len(df_pdf_generated := df_logs_fase1[df_logs_fase1['PDF_Consolidado_Generado'] == True]['MUNICIPIO_GRUPO'].unique())} municipios")
if 'df_logs_fase2' in locals() and not df_logs_fase2.empty:
    mail_ok_count = len(df_logs_fase2[df_logs_fase2['Correo_Preparado'] == True])
    print(f" Correos electrónicos preparados/enviados:    {mail_ok_count} notificaciones")
print(f"---------------------------------------------------------")
print(f" Logs exportados a: {FOLDER_LOGS.name}/")
print(f"   - Operación: LOG_OPERACION.xlsx")
print(f"   - Fallas y Errores: LOG_ERRORES.xlsx")
print(f"   - Resumen Directivo: LOG_RESUMEN.xlsx")
print(f"=========================================================")

# Liberación explícita de memoria RAM residual de la ejecución
try:
    del df_raw, df_procesar, df_rechazados, df_operacion, df_errores_final, df_resumen
    del df_logs_fase1, df_errors_fase1, df_logs_fase2, df_errors_fase2
except NameError:
    pass
gc.collect()
print("Memoria RAM del sistema optimizada. Lote completado.")